# 🎓 DropGuard: Student Dropout Early Warning System
### **Interactive Exploratory Data Analysis, Preprocessing & Multi-Model Benchmarks**
---
This notebook serves as **Member 1's interactive lab** in VS Code / Colab:
1. **Exploratory Data Analysis (EDA):** Inspect distributions, target balance, and check for missing values.
2. **Data Preprocessing:** Categorize 36 features into Numerical, Nominal Categorical, and Binary sets, applying `ColumnTransformer` with `StandardScaler` and `OneHotEncoder`.
3. **Multi-Model Benchmarking:** Train and compare Logistic Regression, Random Forest, and Gradient Boosting on stratified test data.
4. **Feature Importance & XAI:** Extract the primary curricular and socio-economic risk drivers.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("All libraries imported successfully!")

## 1. Load Dataset & Exploratory Analysis

In [ ]:
DATASET_PATH = os.path.join("..", "data", "student_data.csv")

df = pd.read_csv(DATASET_PATH)
# Clean up column whitespace/tabs
df.columns = [c.strip() for c in df.columns]

print(f"Total Students in Dataset: {len(df)}")
print(f"Total Features: {len(df.columns) - 1}")
print(f"Missing Values: {df.isnull().sum().sum()}")

# Display class distribution
print("\n--- Target Distribution ---")
print(df['Target'].value_counts())
df.head()

## 2. Define Preprocessing Taxonomy (`ColumnTransformer`)
We separate features into:
- **Numerical:** Standardized with `StandardScaler`.
- **Categorical (Nominal):** Encoded with `OneHotEncoder(handle_unknown='ignore')` so no false numerical hierarchy is assumed.
- **Binary:** Passthrough as $0/1$ flags.

In [ ]:
NUMERICAL_FEATURES = [
    'Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
    'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)',
    'Unemployment rate', 'Inflation rate', 'GDP'
]

CATEGORICAL_FEATURES = [
    'Marital status', 'Application mode', 'Application order', 'Course',
    'Previous qualification', 'Nacionality', "Mother's qualification",
    "Father's qualification", "Mother's occupation", "Father's occupation"
]

BINARY_FEATURES = [
    'Daytime/evening attendance', 'Displaced', 'Educational special needs',
    'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERICAL_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
        ('bin', 'passthrough', BINARY_FEATURES)
    ]
)

# Binary target formulation: Dropout=1, Graduate=0
binary_df = df[df['Target'].isin(['Dropout', 'Graduate'])].copy()
X = binary_df.drop(columns=['Target'])
y = (binary_df['Target'] == 'Dropout').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Raw Features: {X.shape[1]} -> Preprocessed Encoded Features: {X_train_proc.shape[1]}")

## 3. Multi-Model Benchmark Suite
We evaluate 3 candidate algorithms focusing on **Recall & ROC-AUC**.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=14, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, random_state=42)
}

benchmark_results = []

for name, model in models.items():
    model.fit(X_train_proc, y_train)
    y_pred = model.predict(X_test_proc)
    y_prob = model.predict_proba(X_test_proc)[:, 1]
    
    benchmark_results.append({
        'Model': name,
        'Accuracy': f"{accuracy_score(y_test, y_pred) * 100:.2f}%",
        'Precision (At-Risk)': f"{precision_score(y_test, y_pred) * 100:.2f}%",
        'Recall (At-Risk)': f"{recall_score(y_test, y_pred) * 100:.2f}%",
        'F1-Score': f"{f1_score(y_test, y_pred) * 100:.2f}%",
        'ROC-AUC': f"{roc_auc_score(y_test, y_prob):.4f}"
    })

results_table = pd.DataFrame(benchmark_results)
results_table

## 4. Visual Feature Importance
Inspect which academic & financial factors contribute most to dropout risk.

In [ ]:
# Extract feature names from preprocessor
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_feature_names = NUMERICAL_FEATURES + encoded_cat_features + BINARY_FEATURES

rf_model = models['Random Forest']
feat_importances = pd.Series(rf_model.feature_importances_, index=all_feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_importances.tail(10).plot(kind='barh', color='#1E3A8A', edgecolor='#0284C7')
plt.title('Top 10 Dropout Risk Predictors in College Data', fontsize=12, fontweight='bold')
plt.xlabel('Importance Score')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()